<a href="https://colab.research.google.com/github/kocakcan/ml_foundations/blob/main/intro_to_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
model = BertModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
named_params = list(model.named_parameters())
print("The BERT model has {:} different named parameters.\n".format(len(named_params)))

print("=== Embedding Layer ===\n")
for p in named_params[0:5]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print("\n=== First Encoder ===\n")
for p in named_params[5:21]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print("\n=== Output Layer ===\n")
for p in named_params[-2:]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

The BERT model has 199 different named parameters.

=== Embedding Layer ===

embeddings.word_embeddings.weight                       (30522, 768)
embeddings.position_embeddings.weight                     (512, 768)
embeddings.token_type_embeddings.weight                     (2, 768)
embeddings.LayerNorm.weight                                   (768,)
embeddings.LayerNorm.bias                                     (768,)

=== First Encoder ===

encoder.layer.0.attention.self.query.weight               (768, 768)
encoder.layer.0.attention.self.query.bias                     (768,)
encoder.layer.0.attention.self.key.weight                 (768, 768)
encoder.layer.0.attention.self.key.bias                       (768,)
encoder.layer.0.attention.self.value.weight               (768, 768)
encoder.layer.0.attention.self.value.bias                     (768,)
encoder.layer.0.attention.output.dense.weight             (768, 768)
encoder.layer.0.attention.output.dense.bias                   (768,)
en

In [10]:
# The pooler is a separate linear and tanh activated layer that acts on the [CLS] token's representation
# This pooled_output is often used as a representation for the entire sentence.

In [11]:
# load the bert-base uncased tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
tokenizer.encode("Can finds NLP to be highly intriguing.")

[101, 2064, 4858, 17953, 2361, 2000, 2022, 3811, 23824, 1012, 102]

In [14]:
# run tokens through the model

#1 Turn tokens_with_unknown_words into a tensor (will be size (8,))
#2 Unsquueze a first dimension to simulate batches. Resulting shape is (1, 8)
response = model(torch.tensor(tokenizer.encode("Can finds NLP to be highly intriguing.")).unsqueeze(0))

In [16]:
response

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2760, -0.2602, -0.0949,  ..., -0.3765,  0.3917,  0.4869],
         [ 0.0272,  0.0338,  0.5508,  ..., -0.2681,  0.4822,  0.3296],
         [-0.9126, -0.0358,  0.3538,  ..., -0.7194,  0.4401, -0.2475],
         ...,
         [ 0.2202,  0.1291,  0.3081,  ..., -0.3098,  0.1109, -0.0429],
         [-0.0573, -0.2961, -0.3703,  ...,  0.4762, -0.0134, -0.4318],
         [ 0.5397, -0.0829, -0.3930,  ...,  0.2242, -0.6757, -0.3840]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-7.9536e-01, -2.2278e-01, -6.4746e-01,  6.0603e-01,  4.7414e-01,
          2.2248e-02,  7.3109e-01,  2.7776e-01, -4.7601e-01, -9.9996e-01,
         -3.2936e-02,  3.5850e-01,  9.6317e-01,  1.6054e-01,  8.0588e-01,
         -4.7449e-01,  1.1769e-01, -4.9254e-01,  1.9543e-01, -3.9526e-01,
          5.4644e-01,  9.9966e-01,  2.2856e-01,  2.3224e-01,  2.7254e-01,
          7.4438e-01, -5.5636e-01,  8.5589e-01,  9.3338e-01,  7.630

In [17]:
response.last_hidden_state

tensor([[[-0.2760, -0.2602, -0.0949,  ..., -0.3765,  0.3917,  0.4869],
         [ 0.0272,  0.0338,  0.5508,  ..., -0.2681,  0.4822,  0.3296],
         [-0.9126, -0.0358,  0.3538,  ..., -0.7194,  0.4401, -0.2475],
         ...,
         [ 0.2202,  0.1291,  0.3081,  ..., -0.3098,  0.1109, -0.0429],
         [-0.0573, -0.2961, -0.3703,  ...,  0.4762, -0.0134, -0.4318],
         [ 0.5397, -0.0829, -0.3930,  ...,  0.2242, -0.6757, -0.3840]]],
       grad_fn=<NativeLayerNormBackward0>)

In [18]:
response.pooler_output.shape

torch.Size([1, 768])

In [19]:
model.pooler

BertPooler(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): Tanh()
)

In [20]:
# grab the final encoder's representation of the CLS token
CLS_embedding = response.last_hidden_state[:, 0, :].unsqueeze(0)
CLS_embedding.shape

torch.Size([1, 1, 768])

In [21]:
model.pooler(CLS_embedding).shape

torch.Size([1, 768])

In [22]:
# Running the embedding for CLS through the pooler gives the same output as the `pooler_output`
(model.pooler(CLS_embedding) == response.pooler_output).all()

tensor(True)

In [23]:
total_params = 0
for p in model.parameters():
  if len(p.shape) == 2:
    total_params += p.shape[0] * p.shape[1]
print(f"Total parameters: {total_params:,}")

Total parameters: 109,360,128


In [25]:
"Can" in tokenizer.vocab

False

In [26]:
# BERT's tokenizer is great at handling tokens that are OOV (out of vocabulary) by breaking them up into smaller chunks of known tokens

In [29]:
tokenizer.encode("I love my pet Python.")

[101, 1045, 2293, 2026, 9004, 18750, 1012, 102]

In [30]:
tokenizer.encode("I love coding in Python.")

[101, 1045, 2293, 16861, 1999, 18750, 1012, 102]

In [31]:
# The token `python` will end up with a vector representation from each sentence via BERT.
# What's interesting is that the vector representation `python` will be different for each sentence
# because of the surrounding words in the sentence

In [34]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"Length of BERT base vocabulary: {len(tokenizer.vocab)}")

Length of BERT base vocabulary: 30522


In [36]:
text = "A simple sentence!"
# get token ids per BERT-base's vocabulary
tokens = tokenizer.encode(text)
print(tokens)

[101, 1037, 3722, 6251, 999, 102]


In [37]:
# decode will re-construct the sentence with the added [CLS] and [SEP] tokens
tokenizer.decode(tokens)

'[CLS] a simple sentence! [SEP]'

In [38]:
text = "My friend told me about this class and I love it so far! She was right."

tokens = tokenizer.encode(text)
print(tokens)

[101, 2026, 2767, 2409, 2033, 2055, 2023, 2465, 1998, 1045, 2293, 2009, 2061, 2521, 999, 2016, 2001, 2157, 1012, 102]


In [39]:
# A nicer printout of token ids and token strings
for t in tokens:
  print(f"Token: {t}, subword: {tokenizer.decode([t])}")

Token: 101, subword: [CLS]
Token: 2026, subword: my
Token: 2767, subword: friend
Token: 2409, subword: told
Token: 2033, subword: me
Token: 2055, subword: about
Token: 2023, subword: this
Token: 2465, subword: class
Token: 1998, subword: and
Token: 1045, subword: i
Token: 2293, subword: love
Token: 2009, subword: it
Token: 2061, subword: so
Token: 2521, subword: far
Token: 999, subword: !
Token: 2016, subword: she
Token: 2001, subword: was
Token: 2157, subword: right
Token: 1012, subword: .
Token: 102, subword: [SEP]
